In [2]:
import sympy as sp
from sympy import symbols, diff, solve
import numpy as np

In [3]:
# Define Symbols
psi_dot, m, p, C_d, A, V_ref, V_y, F_x, e_vx, e_psi, e_y = symbols(
    "psi_dot m p C_d A V_ref V_y F_x e_vx e_psi e_y"
)
C_alp_f, C_alp_r, L_f, L_r, delta, I2, K = symbols("C_alp_f C_alp_r L_f L_r delta I2 K")

# State & Control Vectors
x = sp.Matrix([e_vx, V_y, psi_dot, e_y, e_psi]) 
u = sp.Matrix([F_x, delta]) 

# Longitudinal Velocity Error Dynamics
f1 = psi_dot * V_y + 1 / m * (F_x - 0.5 * p * C_d * A * (V_ref + e_vx) ** 2)

# Lateral Velocity Error Dynamics (Removed trailing *)
f2 = -psi_dot * (V_ref + e_vx) + 1 / m * (
    -C_alp_f * ((V_y + L_f * psi_dot) / (V_ref + e_vx) - delta)
    - C_alp_r * ((V_y - L_r * psi_dot) / (V_ref + e_vx))
)

# Yaw Rate Error Dynamics (Removed trailing * and fixed parenthetical grouping)
f3 = 1 / I2 * (
    L_f * -C_alp_f * ((V_y + L_f * psi_dot) / (V_ref + e_vx) - delta)
    - L_r * -C_alp_r * ((V_y - L_r * psi_dot) / (V_ref + e_vx))
)

# CTE Kinematics
f4 = (V_ref + e_vx) * sp.sin(e_psi) + V_y * sp.cos(e_psi)

# Heading Error Kinematics
f5 = psi_dot - K * (e_vx + V_ref)

# State Dynamics Vector
f = sp.Matrix([f1, f2, f3, f4, f5])

In [4]:
# Compute Jacobians
A_matrix = f.jacobian(x)
B_matrix = f.jacobian(u)
# Display Results
print("A Matrix (State Jacobian):")
sp.pprint(A_matrix)
print("\nB Matrix (Control Jacobian):")
sp.pprint(B_matrix)


A Matrix (State Jacobian):
⎡               -0.5⋅A⋅C_d⋅p⋅(2⋅V_ref + 2⋅eᵥₓ)                                 ↪
⎢               ───────────────────────────────                            ψ_d ↪
⎢                              m                                               ↪
⎢                                                                              ↪
⎢         C_alp_f⋅(L_f⋅ψ_dot + V_y)   Cₐₗₚ ᵣ⋅(-Lᵣ⋅ψ_dot + V_y)                 ↪
⎢         ───────────────────────── + ────────────────────────      C_alp_f    ↪
⎢                           2                           2       - ───────────  ↪
⎢              (V_ref + eᵥₓ)               (V_ref + eᵥₓ)          V_ref + eᵥₓ  ↪
⎢-ψ_dot + ────────────────────────────────────────────────────  ────────────── ↪
⎢                                  m                                         m ↪
⎢                                                                              ↪
⎢ C_alp_f⋅L_f⋅(L_f⋅ψ_dot + V_y)   Cₐₗₚ ᵣ⋅Lᵣ⋅(-Lᵣ⋅ψ_dot + V_y)                  ↪
⎢

In [5]:
# Evaluate at equilibrium point (e_vx=0, V_y=0, psi_dot=0, e_y=0, e_psi=0, F_x=0, delta=0)
equilibrium_point = {
    e_vx: 0,
    V_y: 0,
    psi_dot: 0,
    e_y: 0,
    e_psi: 0,
    F_x: 0,
    delta: 0,
}
A_eq = A_matrix.subs(equilibrium_point)
B_eq = B_matrix.subs(equilibrium_point)
print("\nA Matrix at Equilibrium:")
sp.pprint(A_eq)
print("\nB Matrix at Equilibrium:")
sp.pprint(B_eq)


A Matrix at Equilibrium:
⎡-1.0⋅A⋅C_d⋅V_ref⋅p                                                            ↪
⎢───────────────────              0                              0             ↪
⎢         m                                                                    ↪
⎢                                                                              ↪
⎢                          C_alp_f   Cₐₗₚ ᵣ                 C_alp_f⋅L_f   Cₐₗₚ ↪
⎢                        - ─────── - ──────               - ─────────── + ──── ↪
⎢                           V_ref    V_ref                     V_ref        V_ ↪
⎢         0              ──────────────────      -V_ref + ──────────────────── ↪
⎢                                m                                    m        ↪
⎢                                                                              ↪
⎢                                                                2             ↪
⎢                       C_alp_f⋅L_f   Cₐₗₚ ᵣ⋅Lᵣ       C_alp_f⋅L_f    Cₐₗₚ ᵣ⋅Lᵣ ↪
⎢ 

## Skidpad Event

In [6]:
# 1. Define our known Skidpad equilibrium constraints
# We know e_vx is 0, so we solve f5 to find the required steady-state yaw rate
psi_dot_eq = sp.solve(f5.subs(e_vx, 0), psi_dot)[0]

# 2. Substitute those knowns (e_vx = 0 and psi_dot_eq) into f2 and f3
f2_corner = f2.subs({e_vx: 0, psi_dot: psi_dot_eq})
f3_corner = f3.subs({e_vx: 0, psi_dot: psi_dot_eq})

# 3. Solve the remaining system simultaneously for V_y and delta
# This returns a dictionary of the solutions
corner_sols = sp.solve([f2_corner, f3_corner], [V_y, delta])

# 4. Safely extract the results into brand new variables
V_y_eq = sp.simplify(corner_sols[V_y])
delta_eq = sp.simplify(corner_sols[delta])

# Display the final expressions
print("Steady-State Yaw Rate (psi_dot_eq):")
sp.pprint(psi_dot_eq)

print("\nSteady-State Lateral Velocity (V_y_eq):")
sp.pprint(V_y_eq)

print("\nSteady-State Steering Angle (delta_eq):")
sp.pprint(delta_eq)

Steady-State Yaw Rate (psi_dot_eq):
K⋅V_ref

Steady-State Lateral Velocity (V_y_eq):
        ⎛                         2            2  ⎞
K⋅V_ref⋅⎝Cₐₗₚ ᵣ⋅L_f⋅Lᵣ + Cₐₗₚ ᵣ⋅Lᵣ  - L_f⋅V_ref ⋅m⎠
───────────────────────────────────────────────────
                 Cₐₗₚ ᵣ⋅(L_f + Lᵣ)                 

Steady-State Steering Angle (delta_eq):
  ⎛                  2                                              2          ↪
K⋅⎝C_alp_f⋅Cₐₗₚ ᵣ⋅L_f  + 2⋅C_alp_f⋅Cₐₗₚ ᵣ⋅L_f⋅Lᵣ + C_alp_f⋅Cₐₗₚ ᵣ⋅Lᵣ  - C_alp_ ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                            C_alp_f⋅Cₐₗₚ ᵣ⋅(L_f + Lᵣ)          ↪

↪            2                    2  ⎞
↪ f⋅L_f⋅V_ref ⋅m + Cₐₗₚ ᵣ⋅Lᵣ⋅V_ref ⋅m⎠
↪ ────────────────────────────────────
↪                                     


In [7]:
skidpad_equilibrium = {
    e_vx: 0,
    psi_dot: psi_dot_eq,
    V_y: V_y_eq,
    delta: delta_eq,
    e_y: 0,
    e_psi: 0,
}

A_skidpad = sp.simplify(A_matrix.subs(skidpad_equilibrium))
B_skidpad = sp.simplify(B_matrix.subs(skidpad_equilibrium)) 

print("\nA Matrix at Skidpad Equilibrium:")
sp.pprint(A_skidpad)    
print("\nB Matrix at Skidpad Equilibrium:")
sp.pprint(B_skidpad)  



A Matrix at Skidpad Equilibrium:
⎡                                                                              ↪
⎢                                                           -1.0⋅A⋅C_d⋅V_ref⋅p ↪
⎢                                                           ────────────────── ↪
⎢                                                                    m         ↪
⎢                                                                              ↪
⎢  ⎛                  2                                              2         ↪
⎢K⋅⎝C_alp_f⋅Cₐₗₚ ᵣ⋅L_f  + 2⋅C_alp_f⋅Cₐₗₚ ᵣ⋅L_f⋅Lᵣ + C_alp_f⋅Cₐₗₚ ᵣ⋅Lᵣ  - C_alp ↪
⎢───────────────────────────────────────────────────────────────────────────── ↪
⎢                                                        Cₐₗₚ ᵣ⋅V_ref⋅m⋅(L_f + ↪
⎢                                                                              ↪
⎢                ⎛                  2                                          ↪
⎢          K⋅L_f⋅⎝C_alp_f⋅Cₐₗₚ ᵣ⋅L_f  + 2⋅C_alp_f⋅Cₐₗₚ ᵣ⋅L_f⋅Lᵣ + C_alp_f⋅C

## Acceleration event 

In [8]:
accel_equallibrium = {
    e_vx: 0,
    V_y: 0,
    psi_dot: 0,
    e_y: 0,
    e_psi: 0,
    F_x: 0,
    delta: 0,
}
vt_ref = sp.symbols("v_ref(t)")
A_accel = sp.simplify(A_matrix.subs(accel_equallibrium).subs(K, 0).subs(V_ref, vt_ref))  # Set K to 0 for straight-line equilibrium
B_accel = sp.simplify(B_matrix.subs(accel_equallibrium).subs(K, 0).subs(V_ref, vt_ref))  # Set K to 0 for straight-line equilibrium
print("\nA Matrix at Acceleration Equilibrium:")
sp.pprint(A_accel)
print("\nB Matrix at Acceleration Equilibrium:")
sp.pprint(B_accel)


A Matrix at Acceleration Equilibrium:
⎡-1.0⋅A⋅C_d⋅p⋅v_ref(t)                                                         ↪
⎢──────────────────────             0                                0         ↪
⎢          m                                                                   ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                           -C_alp_f - Cₐₗₚ ᵣ      -C_alp_f⋅L_f + Cₐₗₚ ᵣ⋅Lᵣ -  ↪
⎢          0                ─────────────────      ─────────────────────────── ↪
⎢                              m⋅v_ref(t)                        m⋅v_ref(t)    ↪
⎢                                                                              ↪
⎢                                                                    2         ↪
⎢                        -C_alp_f⋅L_f + Cₐₗₚ ᵣ⋅Lᵣ       - C_alp_f⋅L_f  - Cₐₗₚ  ↪
⎢          0             ────────────────────────       ──────────────

In [9]:
# Write everything to a .tex file for LaTeX rendering
with open("results/skidpad_equilibrium.tex", "w") as f:
    f.write("% Skidpad Equilibrium Analysis\n")
    f.write("\\documentclass{article}\n")
    f.write("\\usepackage{amsmath}\n")
    f.write("\\usepackage{amssymb}\n")
    f.write("\\begin{document}\n")
    
    f.write("\\section*{A and B Matrices}\n")
    f.write("\\subsection*{A Matrix}\n")
    f.write(f"\\begin{{equation}}\nA = {sp.latex(A_matrix)}\n\\end{{equation}}\n")
    f.write("\\subsection*{B Matrix}\n")
    f.write(f"\\begin{{equation}}\nB = {sp.latex(B_matrix)}\n\\end{{equation}}\n") 
    
    f.write("\\section*{Skidpad Equilibrium Analysis}\n")
    
    f.write("\\subsection*{Steady-State Yaw Rate (\\(\\psi_{dot,eq}\\))}\n")
    f.write(f"\\begin{{equation}}\n\\psi_{{dot,eq}} = {sp.latex(psi_dot_eq)}\n\\end{{equation}}\n")
    
    f.write("\\subsection*{Steady-State Lateral Velocity (\\(V_{y,eq}\\))}\n")
    f.write(f"\\begin{{equation}}\nV_{{y,eq}} = {sp.latex(V_y_eq)}\n\\end{{equation}}\n")
    
    f.write("\\subsection*{Steady-State Steering Angle (\\(\\delta_{eq}\\))}\n")
    f.write(f"\\begin{{equation}}\n\\delta_{{eq}} = {sp.latex(delta_eq)}\n\\end{{equation}}\n")
    
    f.write("\\subsection*{A Matrix at Skidpad Equilibrium}\n")
    f.write(f"\\begin{{equation}}\nA_{{skidpad}} = {sp.latex(A_skidpad)}\n\\end{{equation}}\n")
    
    f.write("\\subsection*{B Matrix at Skidpad Equilibrium}\n")
    f.write(f"\\begin{{equation}}\nB_{{skidpad}} = {sp.latex(B_skidpad)}\n\\end{{equation}}\n")
    
    f.write("\\end{document}\n")

In [10]:
# Calculate A_straight and B_straight at the straight-line equilibrium point
A_straight = sp.simplify(A_eq.subs(K, 0))  # Set K to 0 for straight-line equilibrium
B_straight = sp.simplify(B_eq.subs(K, 0))  # Set K to 0 for straight-line equilibrium

print("\nA Matrix at Straight-Line Equilibrium:")
sp.pprint(A_straight)
print("\nB Matrix at Straight-Line Equilibrium:")
sp.pprint(B_straight)


A Matrix at Straight-Line Equilibrium:
⎡-1.0⋅A⋅C_d⋅V_ref⋅p                                                            ↪
⎢───────────────────             0                               0             ↪
⎢         m                                                                    ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                        -C_alp_f - Cₐₗₚ ᵣ      -C_alp_f⋅L_f + Cₐₗₚ ᵣ⋅Lᵣ - V_r ↪
⎢         0              ─────────────────      ────────────────────────────── ↪
⎢                             V_ref⋅m                         V_ref⋅m          ↪
⎢                                                                              ↪
⎢                                                                2             ↪
⎢                     -C_alp_f⋅L_f + Cₐₗₚ ᵣ⋅Lᵣ      - C_alp_f⋅L_f  - Cₐₗₚ ᵣ⋅Lᵣ ↪
⎢         0           ────────────────────────      ─────────────────

## Controllability 

In [11]:
## Define Dummy Dictionary for Substitution
m_val = 1000  # kg
V_val = 20    # m/s
C_alp_f_val = 5   # Front cornering stiffness (N/rad)
C_alp_r_val = 5   # Rear cornering stiffness (N/rad)
L_f_val = 1.5    # Distance from CG to front axle (m)
L_r_val = 1.5    # Distance from CG to rear axle (m)
I2_val = 1500   # Yaw moment of inertia (kg*m^2)
K_val = 0.1    # Path curvature (1/m)
p_val = 1.225  # Air density (kg/m^3)
C_d_val = 0.3   # Drag coefficient
A_val = 2.0     # Frontal area (m^2)

# ==========================================
# 1. T37 Vehicle Parameters (from vehicle_config.py)
# ==========================================
m_val = 210.0                           # Mass [kg]
L_val = 1.545                           # Wheelbase [m]
L_f_val = (L_val / 2.0)+0.025                       # CG to front axle [m] (0.7725)
L_r_val = (L_val / 2.0)-0.025                       # CG to rear axle [m] (0.7725)
C_alpha_f_val = 400.0 * (180.0 / np.pi) # Front cornering stiffness [N/rad] (~22918.3)
C_alpha_r_val = 400.0 * (180.0 / np.pi) # Rear cornering stiffness [N/rad] (~22918.3)
I_z_val = (1.0 / 12.0) * m_val * L_val**2       # Yaw moment of inertia [kg*m^2] (~41.77)
rho = 1.225                         # Air density [kg/m^3]
C_d_val = 0.35                          # Drag coefficient
A_val = 1.82                            # Frontal area [m^2]


dummy_subs = {
    m: m_val,
    V_ref: V_val, 
    C_alp_f: C_alpha_f_val, 
    C_alp_r: C_alpha_r_val, 
    L_f: L_f_val, 
    L_r: L_r_val, 
    I2: I_z_val, 
    K: K_val, 
    p: rho, 
    C_d: C_d_val, 
    A: A_val
    }
v_ref_vals = np.linspace(1.0, 45.0, 100) # m/s


In [12]:
## Construct controllability matrix at straight-line equilibrium
C_straight = sp.Matrix.hstack(B_straight,
                            A_straight * B_straight,
                            A_straight**2 * B_straight,
                            A_straight**3 * B_straight,
                            A_straight**4 * B_straight)
C_straight_rank = C_straight.rank()
print("\nControllability Matrix at Straight-Line Equilibrium:")
sp.pprint(C_straight)
print(f"\nRank of Controllability Matrix: {C_straight_rank}")   

## Construct controllability matrix at skidpad equilibrium

A_skidpad_sub = A_skidpad.subs(dummy_subs)
B_skidpad_sub = B_skidpad.subs(dummy_subs)

## Construct controllability matrix at skidpad equilibrium
C_skidpad = sp.Matrix.hstack(B_skidpad_sub,
                            A_skidpad_sub * B_skidpad_sub,
                            A_skidpad_sub**2 * B_skidpad_sub,
                            A_skidpad_sub**3 * B_skidpad_sub,
                            A_skidpad_sub**4 * B_skidpad_sub)
C_skidpad_rank = C_skidpad.rank()
print("\nControllability Matrix at Skidpad Equilibrium:")
sp.pprint(C_skidpad)
print(f"\nRank of Controllability Matrix: {C_skidpad_rank}")

## Construct the controllability matrix at acceleration equilibrium
for v in v_ref_vals:
    A_accel_sub = A_accel.subs(dummy_subs).subs(vt_ref, v)
    B_accel_sub = B_accel.subs(dummy_subs).subs(vt_ref, v)
    C_accel = sp.Matrix.hstack(B_accel_sub,
                                A_accel_sub * B_accel_sub,
                                A_accel_sub**2 * B_accel_sub,
                                A_accel_sub**3 * B_accel_sub,
                                A_accel_sub**4 * B_accel_sub)
    C_accel_rank = C_accel.rank()
    # print("\nControllability Matrix at Acceleration Equilibrium:")
    # sp.pprint(C_accel)
    print(f"\nRank of Controllability Matrix: {C_accel_rank}")  


Controllability Matrix at Straight-Line Equilibrium:
⎡                                                                              ↪
⎢1               -1.0⋅A⋅C_d⋅V_ref⋅p                                            ↪
⎢─       0       ───────────────────                                         0 ↪
⎢m                        2                                                    ↪
⎢                        m                                                     ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢     C_alp_f                         C_alp_f⋅(-C_alp_f - Cₐₗₚ ᵣ)   C_alp_f⋅L_ ↪
⎢0    ───────             0           ─────────────────────────── + ────────── ↪
⎢        m                                            2                        ↪
⎢                                              V_ref⋅m                         ↪
⎢                                                      

In [13]:
## Save all controllability results to a .tex file for LaTeX rendering
with open("results/controllability_results.tex", "w") as f:
    f.write("% Controllability Analysis Results\n")
    f.write("\\documentclass{article}\n")
    f.write("\\usepackage{amsmath}\n")
    f.write("\\usepackage{amssymb}\n")
    f.write("\\begin{document}\n")
    
    f.write("\\section*{Controllability Analysis}\n")
    
    f.write("\\subsection*{Straight-Line Equilibrium}\n")
    f.write(f"\\begin{{equation}}\nC_{{straight}} = {sp.latex(C_straight)}\n\\end{{equation}}\n")
    f.write(f"Rank: {C_straight_rank}\n")
    
    f.write("\\subsection*{Skidpad Equilibrium}\n")
    f.write(f"\\begin{{equation}}\nC_{{skidpad}} = {sp.latex(C_skidpad)}\n\\end{{equation}}\n")
    f.write(f"Rank: {C_skidpad_rank}\n")
    
    f.write("\\subsection*{Acceleration Equilibrium}\n")
    for v in v_ref_vals:
        A_accel_sub = A_accel.subs(dummy_subs).subs(vt_ref, v)
        B_accel_sub = B_accel.subs(dummy_subs).subs(vt_ref, v)
        C_accel = sp.Matrix.hstack(B_accel_sub,
                                    A_accel_sub * B_accel_sub,
                                    A_accel_sub**2 * B_accel_sub,
                                    A_accel_sub**3 * B_accel_sub,
                                    A_accel_sub**4 * B_accel_sub)
        C_accel_rank = C_accel.rank()
        f.write(f"\\textbf{{Velocity: {v} m/s}}\\\\\n")
        # f.write(f"\\begin{{equation}}\nC_{{accel}} = {sp.latex(C_accel)}\n\\end{{equation}}\n")
        f.write(f"Rank: {C_accel_rank}\\\\\n")
    
    f.write("\\end{document}\n")


## Observability

In [14]:
## Define the output vector for observability analysis
y_obs = sp.Matrix([e_vx, psi_dot, e_y, e_psi])  # Observing longitudinal velocity error, lateral position error, heading error, and lateral velocity error
C_obs = sp.Matrix([[1, 0, 0, 0, 0],
                  [0, 0, 1, 0, 0],
                  [0, 0, 0, 1, 0],
                  [0, 0, 0, 0, 1]])
D_obs = sp.zeros(4, 2)  # Assuming no direct feedthrough from control to output

## Create the observability matrix at straight-line equilibrium
O_straight = sp.Matrix.vstack(C_obs,
                            C_obs * A_straight,
                            C_obs * A_straight**2,
                            C_obs * A_straight**3,
                            C_obs * A_straight**4)
O_straight_rank = O_straight.rank()
print("\nObservability Matrix at Straight-Line Equilibrium:")
sp.pprint(O_straight)
print(f"\nRank of Observability Matrix: {O_straight_rank}") 

## Create the observability matrix at skidpad equilibrium
O_skidpad = sp.Matrix.vstack(C_obs,
                            C_obs * A_skidpad_sub,
                            C_obs * A_skidpad_sub**2,
                            C_obs * A_skidpad_sub**3,
                            C_obs * A_skidpad_sub**4)
O_skidpad_rank = O_skidpad.rank()
print("\nObservability Matrix at Skidpad Equilibrium:")
sp.pprint(O_skidpad)
print(f"\nRank of Observability Matrix: {O_skidpad_rank}")

## Create the observability matrix at acceleration equilibrium
for v in v_ref_vals:
    A_accel_sub = A_accel.subs(dummy_subs).subs(vt_ref, v)
    O_accel = sp.Matrix.vstack(C_obs,
                                C_obs * A_accel_sub,
                                C_obs * A_accel_sub**2,
                                C_obs * A_accel_sub**3,
                                C_obs * A_accel_sub**4)
    O_accel_rank = O_accel.rank()
    # print("\nObservability Matrix at Acceleration Equilibrium:")
    # sp.pprint(O_accel)
    print(f"\nRank of Observability Matrix: {O_accel_rank}")    


Observability Matrix at Straight-Line Equilibrium:
⎡           1                                                                  ↪
⎢                                                                              ↪
⎢           0                                                                  ↪
⎢                                                                              ↪
⎢           0                                                                  ↪
⎢                                                                              ↪
⎢           0                                                                  ↪
⎢                                                                              ↪
⎢  -1.0⋅A⋅C_d⋅V_ref⋅p                                                          ↪
⎢  ───────────────────                                                         ↪
⎢           m                                                                  ↪
⎢                                                        

In [15]:
## Save all observability results to a .tex file for LaTeX rendering
with open("results/observability_analysis.tex", "w") as f:
    f.write("% Observability Analysis\n")
    f.write("\\documentclass{article}\n")
    f.write("\\usepackage{amsmath}\n")
    f.write("\\usepackage{amssymb}\n")
    f.write("\\begin{document}\n")
    
    f.write("\\section*{Observability Matrices}\n")
    
    f.write("\\subsection*{Observability Matrix at Straight-Line Equilibrium}\n")
    f.write(f"\\begin{{equation}}\nO_{{straight}} = {sp.latex(O_straight)}\n\\end{{equation}}\n")
    f.write(f"Rank: {O_straight_rank}\n")
    
    f.write("\\subsection*{Observability Matrix at Skidpad Equilibrium}\n")
    f.write(f"\\begin{{equation}}\nO_{{skidpad}} = {sp.latex(O_skidpad)}\n\\end{{equation}}\n")
    f.write(f"Rank: {O_skidpad_rank}\n")
    
    f.write("\\subsection*{Observability Matrix at Acceleration Equilibrium}\n")
    for v in v_ref_vals:
        A_accel_sub = A_accel.subs(dummy_subs).subs(vt_ref, v)
        O_accel = sp.Matrix.vstack(C_obs,
                                    C_obs * A_accel_sub,
                                    C_obs * A_accel_sub**2,
                                    C_obs * A_accel_sub**3,
                                    C_obs * A_accel_sub**4)
        O_accel_rank = O_accel.rank()
        # f.write(f"\\begin{{equation}}\nO_{{accel}}(V_{{ref}}={v}) = {sp.latex(O_accel)}\n\\end{{equation}}\n")
        f.write(f"Rank at V_ref={v} m/s: {O_accel_rank}\n")
    
    f.write("\\end{document}\n")

## Stability

In [16]:
## Check the eigenvalues of A_straight and A_skidpad to assess stability
A_straight_sub = A_straight.subs(dummy_subs)
eigenvals_straight = A_straight_sub.eigenvals()
eigenvals_skidpad = A_skidpad_sub.eigenvals()
print("\nEigenvalues of A Matrix at Straight-Line Equilibrium:")
sp.pprint(eigenvals_straight)
print("\nEigenvalues of A Matrix at Skidpad Equilibrium:")
sp.pprint(eigenvals_skidpad)    

## Check the eigenvalues of A_accel to assess stability at acceleration equilibrium
A_accel_eigenvals = {}
for v in v_ref_vals:
    eigenvals = A_accel_sub.subs(vt_ref, v).eigenvals()
    A_accel_eigenvals[v] = eigenvals
print("\nEigenvalues of A Matrix at Acceleration Equilibrium for Various Speeds:")
for v, eigenvals in A_accel_eigenvals.items():
    print(f"V_ref = {v} m/s:")
    sp.pprint(eigenvals)
    print()


## Calulating the dampening ratio and natural frequency for the dominant eigenvalues at skidpad equilibrium
dominant_eigenvals_skidpad = sorted(eigenvals_skidpad, key=lambda ev: abs(ev), reverse=True)[:2]  # Get the 2 eigenvalues with the largest magnitude
print("\nDominant Eigenvalues at Skidpad Equilibrium:")
sp.pprint(dominant_eigenvals_skidpad)




Eigenvalues of A Matrix at Straight-Line Equilibrium:
{-33.9802049897338: 1, -9.70801226936363: 1, -0.0743166666666667: 1, 0: 1, 1.4 ↪

↪ 4265290902902e-129: 1}

Eigenvalues of A Matrix at Skidpad Equilibrium:
{-33.9371882984066 + 6.55435007978608e-65⋅ⅈ: 1, -8.20909460488364 - 3.03368036 ↪

↪ 891173e-65⋅ⅈ: 1, -1.61625102247389 - 4.45545823047242e-66⋅ⅈ: 1, -4.212456677 ↪

↪ 08031e-32 + 3.56023796402486e-35⋅ⅈ: 1, 4.21245667708031e-32 - 3.560237964024 ↪

↪ 86e-35⋅ⅈ: 1}

Eigenvalues of A Matrix at Acceleration Equilibrium for Various Speeds:
V_ref = 1.0 m/s:
{-16.8573836264159: 1, -2.5596018220718: 1, -0.1672125: 1, 0: 2}

V_ref = 1.4444444444444444 m/s:
{-16.8573836264159: 1, -2.5596018220718: 1, -0.1672125: 1, 0: 2}

V_ref = 1.8888888888888888 m/s:
{-16.8573836264159: 1, -2.5596018220718: 1, -0.1672125: 1, 0: 2}

V_ref = 2.333333333333333 m/s:
{-16.8573836264159: 1, -2.5596018220718: 1, -0.1672125: 1, 0: 2}

V_ref = 2.7777777777777777 m/s:
{-16.8573836264159: 1, -2.5596018220718: 1, -0.

In [17]:
## Save results for stability analysis to a .tex file for LaTeX rendering
with open("results/stability_analysis.tex", "w") as f: 
    f.write("% Stability Analysis Results\n")
    f.write("\\documentclass{article}\n")
    f.write("\\usepackage{amsmath}\n")
    f.write("\\usepackage{amssymb}\n")
    f.write("\\begin{document}\n")
    
    f.write("\\section*{Eigenvalues of A Matrix at Straight-Line Equilibrium}\n")
    f.write(f"\\begin{{equation}}\n\\text{{Eigenvalues}} = {sp.latex(eigenvals_straight)}\n\\end{{equation}}\n")
    
    f.write("\\section*{Eigenvalues of A Matrix at Skidpad Equilibrium}\n")
    f.write(f"\\begin{{equation}}\n\\text{{Eigenvalues}} = {sp.latex(eigenvals_skidpad)}\n\\end{{equation}}\n")
    
    f.write("\\section*{Eigenvalues of A Matrix at Acceleration Equilibrium for Various Speeds}\n")
    for v, eigenvals in A_accel_eigenvals.items():
        f.write(f"\\subsection*{{V_ref = {v} m/s}}\n")
        f.write(f"\\begin{{equation}}\n\\text{{Eigenvalues}} = {sp.latex(eigenvals)}\n\\end{{equation}}\n")
    
    f.write("\\end{document}\n")

In [21]:
from scipy.linalg import solve_continuous_lyapunov

# ... [Continue after your dominant_eigenvals_skidpad block] ...

# 1. Section 3.1: Damping Ratio and Natural Frequency Characterization
print("\n--- 3.1 Skidpad Dynamic Characterization ---")
for i, ev in enumerate(dominant_eigenvals_skidpad):
    # Convert SymPy complex value to a python complex float
    ev_val = complex(ev.evalf())
    
    # Natural Frequency: wn = |lambda|
    wn = np.abs(ev_val)
    
    # Damping Ratio: zeta = -Re(lambda) / wn
    zeta = -ev_val.real / wn if wn > 0 else 0
    
    print(f"Mode {i+1}:")
    print(f"  Eigenvalue: {ev_val:.4f}")
    print(f"  Natural Frequency (wn): {wn:.4f} rad/s")
    print(f"  Damping Ratio (zeta): {zeta:.4f}")
    if zeta < 0.7:
        print("  Observation: Mode is underdamped. LQR must provide active damping.")

# 2. Section 3.2: Controllability Gramian Analysis (Control Effort)
# Converting SymPy matrices to NumPy for numerical linear algebra
B_straight_sub = B_straight.subs(dummy_subs)
A_num = np.array(A_straight_sub.evalf()).astype(np.float64)
B_num = np.array(B_straight_sub.evalf()).astype(np.float64)

# Solve Lyapunov equation for the Controllability Gramian: A*Wc + Wc*A' + BB' = 0
# We use -B*B' because solve_continuous_lyapunov solves AX + XA' + Q = 0
Wc = solve_continuous_lyapunov(A_num, -B_num @ B_num.T)
s_vals = np.linalg.svd(Wc, compute_uv=False)

print("\n--- 3.2 Controllability Energy Analysis ---")
print(f"Gramian Smallest Singular Value: {np.min(s_vals):.2e}")
grammiam_condition_number = np.max(s_vals)/np.min(s_vals) if np.min(s_vals) > 1e-6 else float('Inf')
print(f"Gramian Condition Number: {grammiam_condition_number:.2e}")
print("Takeaway: Small singular values indicate states that are mathematically controllable but physically hard to reach.")

# 3. Section 3.3: Observability Numerical Health (Condition Number)
# Assuming C_obs_sub is defined in your SymPy block
C_num = np.array(C_obs.evalf()).astype(np.float64)

# Re-construct Observability Matrix Numerically for high-precision conditioning check
O_num = np.vstack([C_num @ np.linalg.matrix_power(A_num, i) for i in range(5)])
cond_O = np.linalg.cond(O_num)

print("\n--- 3.3 Observability Numerical Health ---")
print(f"Condition Number of Observability Matrix (O): {cond_O:.2e}")
if cond_O > 1e10 or np.isinf(cond_O):
    print("Warning: High condition number suggests state estimation is sensitive to sensor noise.")
else:
    print("Result: System is well-conditioned for state estimation.")


--- 3.1 Skidpad Dynamic Characterization ---
Mode 1:
  Eigenvalue: -33.9372+0.0000j
  Natural Frequency (wn): 33.9372 rad/s
  Damping Ratio (zeta): 1.0000
Mode 2:
  Eigenvalue: -8.2091-0.0000j
  Natural Frequency (wn): 8.2091 rad/s
  Damping Ratio (zeta): 1.0000

--- 3.2 Controllability Energy Analysis ---
Gramian Smallest Singular Value: 0.00e+00
Gramian Condition Number: inf
Takeaway: Small singular values indicate states that are mathematically controllable but physically hard to reach.

--- 3.3 Observability Numerical Health ---
Condition Number of Observability Matrix (O): 1.27e+06
Result: System is well-conditioned for state estimation.


/home/kpa/miniconda3/lib/python3.13/site-packages/scipy/_lib/_util.py:1181: RuntimeWarning: Input "a" has an eigenvalue pair whose sum is very close to or exactly zero. The solution is obtained via perturbing the coefficients.
  return f(*arrays, *other_args, **kwargs)


In [29]:
C_obs = sp.Matrix([[1, 0, 0, 0, 0],
                  [0, 0, 1, 0, 0],
                  [0, 0, 0, 0, 1]])

## Create the observability matrix at straight-line equilibrium
O_straight_broken = sp.Matrix.vstack(C_obs,
                            C_obs * A_straight_sub,
                            C_obs * A_straight_sub**2,
                            C_obs * A_straight_sub**3,
                            C_obs * A_straight_sub**4)
O_straight_broken_rank = O_straight_broken.rank()
print("\n Observability Matrix with Broken Sensor at Straight-Line Equilibrium rank:")
sp.pprint(O_straight_broken_rank)


O_straight_broken_nullity = O_straight_broken.nullspace()
print("\nNull Space of Observability Matrix with Broken Sensor at Straight-Line Equilibrium:")
for i, vec in enumerate(O_straight_broken_nullity):
    print(f"Null Vector {i+1}:")
    sp.pprint(vec)  


 Observability Matrix with Broken Sensor at Straight-Line Equilibrium rank:
4

Null Space of Observability Matrix with Broken Sensor at Straight-Line Equilibrium:
Null Vector 1:
⎡0⎤
⎢ ⎥
⎢0⎥
⎢ ⎥
⎢0⎥
⎢ ⎥
⎢1⎥
⎢ ⎥
⎣0⎦
